# Pennsylvania walkthrough

End-to-end usage of the `gerrydetect` pipeline on Pennsylvania.
Assumes data has been downloaded and `scripts/build_graph.py pa` has been run.

In [ ]:
import pickle
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROC = Path('../data/processed')
ENS  = Path('../data/ensembles')

with open(PROC / 'pa_graph.pkl', 'rb') as f:
    graph = pickle.load(f)
gdf = gpd.read_parquet(PROC / 'pa_precincts.parquet')

print(graph.number_of_nodes(), 'precincts')
print(graph.number_of_edges(), 'edges')
print(gdf.geometry.crs)

## Run a small ensemble and look at metrics

In [ ]:
from gerrydetect.spectral import recursive_bisect
from gerrydetect.partition import Partition
from gerrydetect.metrics import all_metrics
from gerrydetect import mcmc

k = len({int(graph.nodes[n].get('district', 0)) for n in graph.nodes})
print('k =', k)

seed = recursive_bisect(graph, k=k, pop_tol=0.02)
samples, stats = mcmc.run(graph, seed, n_steps=200, lag=50, burn_in=2000)
print(stats)

In [ ]:
from gerrydetect.analysis import compute_metrics_on_ensemble, outlier_analysis

df = compute_metrics_on_ensemble(samples, gdf=gdf)
df.describe()

In [ ]:
enacted = Partition(graph, {n: int(graph.nodes[n]['district']) for n in graph.nodes})
results = outlier_analysis(all_metrics(enacted, gdf=gdf), df)
for r in results:
    print(f"{r.metric:18s}  enacted={r.enacted_value:.4f}  pct={r.percentile:5.1f}  p={r.p_value_two_sided:.4f}  -> {r.direction}")

In [ ]:
from gerrydetect.viz import plot_histogram, plot_district_map

plot_histogram(df['cut_edge_ratio'].to_numpy(),
               results[0].enacted_value,
               title='PA cut edge ratio',
               xlabel='cut edge ratio',
               p_value=results[0].p_value_two_sided)

In [ ]:
plot_district_map(gdf, enacted.assignment, title='PA enacted CD map')